<a href="https://colab.research.google.com/github/nitinog10/Mini-SLM/blob/main/ollama%20using%20claude%20opus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install unsloth
!pip install --no-deps "xformers<0.0.29" "trl<0.13.0" peft accelerate bitsandbytes

INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.6/209.6 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━ 11.4/24.6 MB 163.4 MB/s eta 0:00:01

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import os

# Import FastLanguageModel for Unsloth models
from unsloth import FastLanguageModel

# 1. Load the dataset
dataset_name = "lordx64/reasoning-distill-claude-opus-4-7-max"
ds = load_dataset(dataset_name, split="train")

# Updated Formatting function for Llama-3 prompt style
def formatting_prompts_func(example):
    instruction = example.get('instruction', '')
    user_input = example.get('input', '')
    output = example.get('output', '')
    text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a reasoning assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{instruction}\n{user_input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{output}<|eot_id|>"
    return {"text": text}

# Map the formatting function
ds = ds.map(formatting_prompts_func)

# 2. Model Configuration: Llama-3.2-1B-Instruct
model_id = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"

# Load model and tokenizer using Unsloth's FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# 4. SFT Configuration - Increased max_steps for better convergence
sft_config = SFTConfig(
    output_dir="./llama-reasoning-results",
    max_steps=200,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_32bit",
    logging_steps=10,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=2048,
)

# 5. Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    tokenizer=tokenizer,
    args=sft_config,
)

trainer.train()

### Inference
Now let's test the model to see its response.

In [ ]:
# 1. Prepare model for fast inference
FastLanguageModel.for_inference(model)

prompt = "Explain the concept of black holes in simple terms."

# Format using the same template used during training
inputs = tokenizer(
    [f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a reasoning assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"],
    return_tensors="pt"
).to("cuda")

# Explicitly set pad_token_id and generate
outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True, pad_token_id=tokenizer.eos_token_id)
response = tokenizer.batch_decode(outputs, skip_special_tokens=False)

# Improved extraction: split by the assistant header and clean up
full_text = response[0]
if "assistant<|end_header_id|>" in full_text:
    generated_text = full_text.split("assistant<|end_header_id|>")[-1].replace("<|eot_id|>", "").strip()
    print(f"Model Output:\n{generated_text}")
else:
    print("Could not find assistant header. Raw output:\n", full_text)

### Interactive Inference UI

Use the input box below to ask the fine-tuned model questions and get its responses.

In [ ]:
from ipywidgets import Textarea, Button, Output, VBox
from IPython.display import display

# Create widgets
question_input = Textarea(
    value='',
    placeholder='Type your question here...',
    description='Question:',
    disabled=False,
    layout={'width': '80%', 'height': '100px'}
)

submit_button = Button(description='Get Response', button_style='primary')
output_area = Output()

def on_button_click(b):
    with output_area:
        output_area.clear_output()
        prompt_text = question_input.value
        if prompt_text:
            print("Generating response... Please wait.")

            # Ensure model is in inference mode
            FastLanguageModel.for_inference(model)

            # Format input
            inputs = tokenizer(
                [f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a reasoning assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{prompt_text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"],
                return_tensors="pt"
            ).to("cuda")

            outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True, pad_token_id=tokenizer.eos_token_id)
            response = tokenizer.batch_decode(outputs)[0]

            # Extract assistant text
            generated_text = response.split("assistant<|end_header_id|>")[-1].replace("<|eot_id|>", "").strip()

            output_area.clear_output()
            print(f"Model Response:\n{generated_text}")
        else:
            print("Please enter a question.")

submit_button.on_click(on_button_click)

# Display widgets
display(VBox([question_input, submit_button, output_area]))